# 01 - Quantization Comparison

FP32, FP16, INT8, INT4 quantization, GPTQ, AWQ, memory vs accuracy trade-offs, benchmark visualization.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded!')


## 1. Quantization Basics


In [ ]:
# Simulate model weights
np.random.seed(42)
original_weights = np.random.randn(1000, 1000).astype(np.float32)
print(f'Original weights shape: {original_weights.shape}')
print(f'Original dtype: {original_weights.dtype}')
print(f'Original size: {original_weights.nbytes / 1024:.2f} KB')


## 2. FP32 vs FP16 vs INT8 vs INT4


In [ ]:
def quantize_fp16(weights):
    return weights.astype(np.float16)

def quantize_int8(weights):
    # Symmetric quantization to INT8
    scale = np.max(np.abs(weights)) / 127
    quantized = np.round(weights / scale).astype(np.int8)
    return quantized, scale

def quantize_int4(weights):
    # Simulate INT4 (store as int8 but only 4 bits used)
    scale = np.max(np.abs(weights)) / 7
    quantized = np.clip(np.round(weights / scale), -8, 7).astype(np.int8)
    return quantized, scale

def dequantize_int8(quantized, scale):
    return quantized.astype(np.float32) * scale

def dequantize_int4(quantized, scale):
    return quantized.astype(np.float32) * scale

fp16_weights = quantize_fp16(original_weights)
int8_weights, int8_scale = quantize_int8(original_weights)
int4_weights, int4_scale = quantize_int4(original_weights)

print('Quantization Results:')
print(f'  FP32: {original_weights.nbytes / 1024:.2f} KB')
print(f'  FP16: {fp16_weights.nbytes / 1024:.2f} KB')
print(f'  INT8: {int8_weights.nbytes / 1024:.2f} KB')
print(f'  INT4: ~{int4_weights.nbytes / 2 / 1024:.2f} KB (packed)')


## 3. GPTQ Block-wise Quantization


In [ ]:
class GPTQQuantizer:
    def __init__(self, block_size=128):
        self.block_size = block_size
    
    def quantize(self, weights):
        rows, cols = weights.shape
        quantized = np.zeros_like(weights, dtype=np.int8)
        scales = []
        for i in range(0, rows, self.block_size):
            block = weights[i:i+self.block_size]
            scale = np.max(np.abs(block)) / 127
            if scale == 0:
                scale = 1e-8
            quantized[i:i+self.block_size] = np.round(block / scale).astype(np.int8)
            scales.append(scale)
        return quantized, scales
    
    def dequantize(self, quantized, scales):
        rows = quantized.shape[0]
        dequantized = np.zeros_like(quantized, dtype=np.float32)
        for i, scale in enumerate(scales):
            start = i * self.block_size
            end = min(start + self.block_size, rows)
            dequantized[start:end] = quantized[start:end].astype(np.float32) * scale
        return dequantized

gptq = GPTQQuantizer(block_size=128)
gptq_quantized, gptq_scales = gptq.quantize(original_weights)
gptq_dequantized = gptq.dequantize(gptq_quantized, gptq_scales)

print(f'GPTQ scales count: {len(gptq_scales)}')
print(f'GPTQ compressed size: ~{gptq_quantized.nbytes / 1024 + len(gptq_scales)*4 / 1024:.2f} KB')


## 4. AWQ Activation-Aware Quantization


In [ ]:
class AWQQuantizer:
    def __init__(self, block_size=128):
        self.block_size = block_size
    
    def quantize(self, weights, activations=None):
        # Simulate activation-aware scaling
        rows, cols = weights.shape
        if activations is None:
            activations = np.random.randn(cols).astype(np.float32)
        
        # Scale weights by activation magnitudes
        act_scale = np.abs(activations)
        act_scale = act_scale / (np.max(act_scale) + 1e-8)
        scaled_weights = weights * act_scale[np.newaxis, :]
        
        quantized = np.zeros_like(weights, dtype=np.int8)
        scales = []
        for i in range(0, rows, self.block_size):
            block = scaled_weights[i:i+self.block_size]
            scale = np.max(np.abs(block)) / 127
            if scale == 0:
                scale = 1e-8
            quantized[i:i+self.block_size] = np.round(block / scale).astype(np.int8)
            scales.append(scale)
        return quantized, scales, act_scale
    
    def dequantize(self, quantized, scales, act_scale):
        rows = quantized.shape[0]
        dequantized = np.zeros_like(quantized, dtype=np.float32)
        for i, scale in enumerate(scales):
            start = i * self.block_size
            end = min(start + self.block_size, rows)
            dequantized[start:end] = quantized[start:end].astype(np.float32) * scale
        dequantized = dequantized / (act_scale[np.newaxis, :] + 1e-8)
        return dequantized

awq = AWQQuantizer(block_size=128)
awq_quantized, awq_scales, awq_act = awq.quantize(original_weights)
awq_dequantized = awq.dequantize(awq_quantized, awq_scales, awq_act)

print(f'AWQ quantization complete!')
print(f'Activation scale shape: {awq_act.shape}')


## 5. Accuracy Comparison


In [ ]:
def compute_error(original, reconstructed):
    mse = np.mean((original - reconstructed) ** 2)
    mae = np.mean(np.abs(original - reconstructed))
    snr = 20 * np.log10(np.std(original) / (np.sqrt(mse) + 1e-10))
    return {'MSE': mse, 'MAE': mae, 'SNR_dB': snr}

# Dequantize all
int8_deq = dequantize_int8(int8_weights, int8_scale)
int4_deq = dequantize_int4(int4_weights, int4_scale)

results = {
    'FP32 (Baseline)': compute_error(original_weights, original_weights),
    'FP16': compute_error(original_weights, fp16_weights.astype(np.float32)),
    'INT8': compute_error(original_weights, int8_deq),
    'INT4': compute_error(original_weights, int4_deq),
    'GPTQ': compute_error(original_weights, gptq_dequantized),
    'AWQ': compute_error(original_weights, awq_dequantized)
}

print('Reconstruction Error Comparison:')
print(f'{'Method':<20} {'MSE':<12} {'MAE':<12} {'SNR (dB)':<12}')
print('-' * 56)
for method, metrics in results.items():
    print(f'{method:<20} {metrics['MSE']:<12.6f} {metrics['MAE']:<12.6f} {metrics['SNR_dB']:<12.2f}')


## 6. Memory vs Accuracy Trade-off


In [ ]:
# Simulate model sizes for different parameter counts
param_counts = [1, 3, 7, 13, 30, 65]  # Billions
methods = ['FP32', 'FP16', 'INT8', 'INT4', 'GPTQ', 'AWQ']
bits_per_param = {'FP32': 32, 'FP16': 16, 'INT8': 8, 'INT4': 4, 'GPTQ': 4.5, 'AWQ': 4.5}

memory_gb = {m: [p * bits_per_param[m] / 8 / 1024 for p in param_counts] for m in methods}

plt.figure(figsize=(12, 5))
colors = {'FP32': 'red', 'FP16': 'orange', 'INT8': 'green', 'INT4': 'blue', 'GPTQ': 'purple', 'AWQ': 'brown'}
for method in methods:
    plt.plot(param_counts, memory_gb[method], marker='o', label=method, color=colors[method], linewidth=2)
plt.xlabel('Model Parameters (Billions)')
plt.ylabel('Memory (GB)')
plt.title('Model Memory by Quantization Method')
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.tight_layout()
plt.show()


## 7. Perplexity Simulation


In [ ]:
# Simulate perplexity vs quantization (lower is better)
methods_perf = ['FP32', 'FP16', 'INT8', 'INT4', 'GPTQ', 'AWQ']
perplexity = [8.5, 8.55, 8.8, 12.5, 9.2, 8.9]
memory = [32, 16, 8, 4, 4.5, 4.5]

fig, ax1 = plt.subplots(figsize=(10, 5))

x = np.arange(len(methods_perf))
bars = ax1.bar(x, perplexity, color='steelblue', alpha=0.7)
ax1.set_xlabel('Quantization Method')
ax1.set_ylabel('Perplexity (lower is better)', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')
ax1.set_xticks(x)
ax1.set_xticklabels(methods_perf)

ax2 = ax1.twinx()
ax2.plot(x, memory, color='red', marker='o', linewidth=2, markersize=8)
ax2.set_ylabel('Memory (GB for 7B model)', color='red')
ax2.tick_params(axis='y', labelcolor='red')

plt.title('Perplexity vs Memory Trade-off (7B Model)')
plt.tight_layout()
plt.show()


## 8. Exercises


In [ ]:
# Exercise 1: Implement mixed-precision quantization
# Exercise 2: Compare quantization-aware training vs post-training
# Exercise 3: Add SmoothQuant integration
# Exercise 4: Benchmark inference speed simulation
print('Exercises listed!')
